# `alt_transformer_data` -- Building Training Sequences for the Transformer

**Ported/adapted from `project_hrishav/train.py`**'s `build_innings_sequences()` and `player_graph.py`'s `PlayerEmbedLookup`. This is the glue module that turns raw ball-by-ball data + a player embedding table into the exact `(features, win_label, nb_labels, score_label)` dict structure `transformer_model.InningsDataset` expects.

**Key design fact worth understanding**: player embeddings here are **precomputed once, up front, and frozen** for the rest of training -- they are baked directly into each ball's feature vector as plain numbers, not looked up dynamically inside the model's forward pass via a trainable `nn.Embedding` layer. This matches hrishav's own original design (`train.py` calls `embed_table.get_all_embeddings().numpy()` once and uses that fixed matrix from then on) -- it's simpler to reason about and avoids needing gradient flow through an embedding lookup during sequence training.

In [1]:
import numpy as np
import pandas as pd
import torch

from src.game_state import build_game_state_matrix, GAME_STATE_DIM
from src.transformer_model import PlayerEmbedTable, PLAYER_EMBED_DIM, NEXT_BALL_MAP

SEED = 42

## `build_player_registry()`

Every player who has ever batted, bowled, or stood as non-striker gets a unique integer ID, starting from **1** (not 0) -- index 0 is deliberately reserved so it can represent "padding/unknown player" in `PlayerEmbedTable`, which uses `padding_idx=0` to guarantee that slot is always the zero vector.

In [2]:
def build_player_registry(df: pd.DataFrame) -> dict[str, int]:
    """Deterministic player_name -> int id map (0 reserved for padding/unknown)."""
    players = sorted(set(df["batter"]) | set(df["bowler"]) | set(df["non_striker"]))
    return {name: i + 1 for i, name in enumerate(players)}

## `build_embedding_lookup()`

Creates a `PlayerEmbedTable` (from `transformer_model`), seeds PyTorch's RNG so the random Xavier initialisation is reproducible, then immediately extracts the raw weight matrix into a plain dict of `{player_name: 32-dim numpy vector}`. From this point on, the `PlayerEmbedTable` module itself is discarded -- only this plain dict is used, which is what makes the embeddings "frozen" for the rest of the pipeline (there's no PyTorch module left to backpropagate into).

In [3]:
def build_embedding_lookup(registry: dict[str, int], seed: int = SEED) -> dict[str, np.ndarray]:
    """Fixed random Xavier-init embedding per player (frozen, not trained further)."""
    torch.manual_seed(seed)
    table = PlayerEmbedTable(num_players=len(registry), embed_dim=PLAYER_EMBED_DIM)
    all_embeds = table.embed.weight.detach().numpy()  # (num_players+1, embed_dim)
    return {name: all_embeds[idx] for name, idx in registry.items()}

## `_encode_next_ball()`

Small private helper mapping a single ball's outcome to one of the 7 next-ball classes: a wicket always wins (checked first, regardless of runs scored off the same delivery e.g. a run-out attempting a bye), otherwise the runs off the bat are mapped via `NEXT_BALL_MAP` (any value not in that map -- e.g. 5 runs, which is rare but does happen -- defaults to the "dot ball" class, matching hrishav's original convention).

In [4]:
def _encode_next_ball(row) -> int:
    if row["is_wicket"] == 1:
        return 6
    return NEXT_BALL_MAP.get(int(row["runs_batter"]), 0)

## `build_innings_sequences()` -- the main entry point

Given the full ball-by-ball DataFrame, a player embedding lookup, and a set of years to include, this builds a Python list of dicts -- one per `(match_id, innings)` -- ready to feed straight into `transformer_model.InningsDataset`.

**Steps:**
1. Filter to the requested years (this is how train/val/test    splits are implemented -- see `run_alt_transformer.py`,    which passes disjoint year sets for each split).
2. Compute the 24-dim game-state matrix for every remaining    ball via `src.game_state.build_game_state_matrix`.
3. Look up each ball's batter/bowler/non-striker embedding    (falling back to an all-zero vector for any name somehow    missing from the lookup -- shouldn't happen in practice    since the registry is built from the same data, but it's a    safe default rather than a `KeyError`).
4. Concatenate game-state + 3 embeddings into the final 120-dim    per-ball feature vector, and assert the dimension is exactly    right (`GAME_STATE_DIM + 3*PLAYER_EMBED_DIM`).
5. Compute the next-ball label for every row, and the two    innings-level labels (`win_label`, `score_label`) via    `groupby(...).transform("first"/"sum")`.
6. Finally, group everything by `(match_id, innings)` and    package each group's rows into one sequence dict.

In [5]:
def build_innings_sequences(
    df: pd.DataFrame, embed_lookup: dict[str, np.ndarray], years: set[int],
) -> list[dict]:
    """
    One sequence per (match_id, innings) with year in `years`.

    Returns a list of dicts: features (T, 120) float32, win_label (0/1),
    nb_labels (T,) int, score_label (float, final_score/250).
    """
    subset = df[df["year"].isin(years)]
    if subset.empty:
        return []

    game_state, d = build_game_state_matrix(subset)
    d = d.reset_index(drop=True)

    bat_emb = np.stack([embed_lookup.get(n, embed_lookup[next(iter(embed_lookup))] * 0) for n in d["batter"]])
    bowl_emb = np.stack([embed_lookup.get(n, embed_lookup[next(iter(embed_lookup))] * 0) for n in d["bowler"]])
    non_emb = np.stack([embed_lookup.get(n, embed_lookup[next(iter(embed_lookup))] * 0) for n in d["non_striker"]])

    features = np.concatenate([game_state, bat_emb, bowl_emb, non_emb], axis=1).astype(np.float32)
    assert features.shape[1] == GAME_STATE_DIM + 3 * PLAYER_EMBED_DIM

    d["nb_label"] = d.apply(_encode_next_ball, axis=1)
    final_scores = d.groupby(["match_id", "innings"])["runs_total"].transform("sum")
    win_labels = d.groupby(["match_id", "innings"])["batting_wins"].transform("first")

    sequences = []
    for (mid, inn), idx in d.groupby(["match_id", "innings"]).groups.items():
        idx = np.asarray(idx)
        sequences.append({
            "features": features[idx],
            "win_label": float(win_labels.iloc[idx[0]]),
            "nb_labels": d["nb_label"].values[idx],
            "score_label": float(final_scores.iloc[idx[0]]) / 250.0,
        })
    return sequences